# 10b. Validation Baseline Recommender

**Role of this notebook.** Train the vanilla behavior-based recommender for the validation experiment. The model predicts simulated behavior heads from scalable edge features, converts those predicted heads into a single recommender score, and ranks the holdout candidates from notebook 10a.

**Steps.**

1. Define the serving-safe feature set and exclude structural/oracle quantities.
2. Load simulated exploration rows for training/validation and holdout candidates for ranking.
3. Encode continuous, binary, and categorical features using train-split statistics and stable category mappings.
4. Train an edge MLP to predict four behavior labels.
5. Convert predicted behavior probabilities into a vanilla score using the calibrated head weights.
6. Rank each user's holdout candidates and compute utility-based top-100 diagnostics.
7. Save the trained checkpoint, holdout scores, top-100 recommendations, and metrics.

This baseline is intentionally behavior-driven: it learns from simulated completion, long-watch, rewatch, and negative labels, not from posterior expected utility or structural estimates.


In [1]:
from pathlib import Path
import json
import math
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
OUTPUT_DIR = PROJECT_ROOT / "python_code_new" / "outputs"
SIM_OUTPUT_DIR = OUTPUT_DIR / "validation_simulation"
GNN_DATA_PATH = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed" / "gnn_data.pt"
HEAD_WEIGHT_PATH = OUTPUT_DIR / "head_weight_pl_weights.json"

EXPLORATION_PATH = SIM_OUTPUT_DIR / "validation_exploration_interactions.parquet"
HOLDOUT_PATH = SIM_OUTPUT_DIR / "validation_holdout_candidates.parquet"
MANIFEST_PATH = SIM_OUTPUT_DIR / "validation_feature_manifest.json"

HOLDOUT_SCORE_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_baseline_holdout_scores.parquet"
TOP100_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_baseline_top100.parquet"
METRICS_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_baseline_metrics.json"
MODEL_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_baseline_mlp.pt"

RANDOM_SEED = 20260624
TRAIN_SESSION_MAX = 159
VALID_SESSION_MIN = 160
TOP_K = 100
BATCH_SIZE = 8192
EVAL_BATCH_SIZE = 131072
NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 10
MIN_VALID_LOSS_DELTA = 1e-5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10
HIDDEN_SIZES = [256, 128, 64]
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

for p in [EXPLORATION_PATH, HOLDOUT_PATH, MANIFEST_PATH, GNN_DATA_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

print("device:", DEVICE)
print("simulation dir:", SIM_OUTPUT_DIR)

device: mps
simulation dir: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation


## 1. Feature Set

The baseline uses only features that could be available at recommendation time: user attributes, video attributes, context variables, and simulated history summaries. It excludes:

| Excluded object | Reason |
|---|---|
| `u_star`, `mu_hat`, `sigma_hat` | oracle utility objects from the simulator |
| posterior variables such as `r_hat`, `tau_hat`, `EU` | structural estimates not available to the vanilla baseline |
| simulated outcomes for the row being scored | labels cannot be used as features |
| belief columns | reserved for the structural estimator rather than the serving baseline |
| `hist_author_recency_days` | calendar-time recency is not meaningful for the simulated holdout ranking task |

The resulting feature list is the serving-safe tabular feature space for the vanilla MLP.


In [2]:
manifest = json.loads(MANIFEST_PATH.read_text())
full_continuous_cols = list(manifest["continuous_cols"])
full_binary_cols = list(manifest["binary_cols"])
full_categorical_cols = list(manifest["categorical_cols"])

DROP_FEATURES = {"hist_author_recency_days"}
FORBIDDEN_FEATURES = {
    "wr_star", "log_wr_star", "completion_star", "long_star", "rewatch_star", "neg_star",
    "watch_ratio", "play_duration", "y_complete", "y_long", "y_rewatch", "y_neg",
    "u_star", "EU", "q_target", "r_hat", "tilde_r", "z_star", "p_accept",
    "tau_it", "mu_hat", "sigma_hat", "c_i", "log_wr_star",
}

continuous_cols = [c for c in full_continuous_cols if c not in DROP_FEATURES]
binary_cols = [c for c in full_binary_cols if c not in DROP_FEATURES]
categorical_cols = [c for c in full_categorical_cols if c not in DROP_FEATURES]
feature_cols = continuous_cols + binary_cols + categorical_cols

for c in feature_cols:
    if c in FORBIDDEN_FEATURES or c.startswith("belief_") or c.startswith("alpha_"):
        raise ValueError(f"Forbidden feature leaked into baseline feature set: {c}")

label_cols = ["y_complete", "y_long", "y_rewatch", "y_neg"]
head_names = ["complete", "long", "rewatch", "neg"]

print("continuous features:", len(continuous_cols))
print("binary features:", len(binary_cols))
print("categorical features:", len(categorical_cols))
print("total features:", len(feature_cols))
print("dropped features:", sorted(DROP_FEATURES))

continuous features: 22
binary features: 6
categorical features: 35
total features: 63
dropped features: ['hist_author_recency_days']


## 2. Train/Validation/Holdout Split

The exploration table contains the simulated random-exploration interactions. These rows provide supervised behavior labels for fitting the baseline.

The split is chronological by simulated session:

| Split | Session indices | Use |
|---|---:|---|
| Train | `0` to `159` | fit MLP weights |
| Validation | `160` to `199` | early stopping and model selection |
| Holdout candidates | separate sampled candidate set | recommendation ranking only |

The holdout rows are never used to fit the model. They are scored only after training is finished.


In [3]:
exploration_cols = sorted(set(["user_id", "video_id", "session_idx", "u_star"] + feature_cols + label_cols))
holdout_cols = sorted(set(["user_id", "video_id", "u_star"] + feature_cols))

exploration = pd.read_parquet(EXPLORATION_PATH, columns=exploration_cols)
holdout = pd.read_parquet(HOLDOUT_PATH, columns=holdout_cols)

missing_train = sorted(set(feature_cols + label_cols + ["session_idx"]) - set(exploration.columns))
missing_holdout = sorted(set(feature_cols + ["u_star"]) - set(holdout.columns))
if missing_train:
    raise KeyError(f"Exploration table missing columns: {missing_train}")
if missing_holdout:
    raise KeyError(f"Holdout table missing columns: {missing_holdout}")

train_mask = exploration["session_idx"].le(TRAIN_SESSION_MAX).to_numpy()
valid_mask = exploration["session_idx"].ge(VALID_SESSION_MIN).to_numpy()
if train_mask.sum() == 0 or valid_mask.sum() == 0:
    raise ValueError("Train/validation split is empty")

print("exploration rows:", f"{len(exploration):,}")
print("train rows:", f"{train_mask.sum():,}")
print("valid rows:", f"{valid_mask.sum():,}")
print("holdout rows:", f"{len(holdout):,}")
print("holdout users:", holdout["user_id"].nunique())

exploration rows: 3,554,000
train rows: 2,843,200
valid rows: 710,800
holdout rows: 1,777,000
holdout users: 1777


## 3. Feature Encoding

The MLP receives three feature blocks:

| Block | Encoding procedure |
|---|---|
| Continuous | standardize using train-split mean and standard deviation |
| Binary | cast to `0/1` float indicators |
| Categorical | map values to integer ids and learn embeddings |

Categorical mappings are aligned to the existing processed-data metadata whenever possible. Unknown or missing categorical values are mapped to `0`, so the holdout scorer can handle values not seen during training.


In [4]:
gnn_data = torch.load(GNN_DATA_PATH, map_location="cpu", weights_only=False)
feature_metadata = gnn_data["feature_metadata"]
cat_meta = feature_metadata["categorical_mappings"]

def encode_continuous(frame, cols, mean=None, std=None):
    if not cols:
        arr = np.zeros((len(frame), 0), dtype=np.float32)
        return arr, np.zeros(0, dtype=np.float32), np.ones(0, dtype=np.float32)
    arr = frame[cols].apply(pd.to_numeric, errors="coerce").astype("float32")
    if mean is None:
        mean = arr.mean(axis=0).to_numpy(dtype=np.float32)
    if std is None:
        std = arr.std(axis=0, ddof=0).replace(0, 1.0).to_numpy(dtype=np.float32)
    arr = arr.fillna(pd.Series(mean, index=cols))
    x = ((arr.to_numpy(dtype=np.float32) - mean) / np.maximum(std, 1e-6)).astype(np.float32)
    return x, mean.astype(np.float32), np.maximum(std, 1e-6).astype(np.float32)

def encode_binary(frame, cols):
    if not cols:
        return np.zeros((len(frame), 0), dtype=np.float32)
    arr = frame[cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).clip(0, 1).astype("float32")
    return arr.to_numpy(dtype=np.float32)

def encode_categorical(frame, cols):
    if not cols:
        return np.zeros((len(frame), 0), dtype=np.int64), []
    encoded = []
    cardinalities = []
    for col in cols:
        info = cat_meta[col]
        mapping = info["mapping"]
        cardinalities.append(int(info["cardinality"]))
        codes = frame[col].astype("string").map(mapping).fillna(0).astype("int64").to_numpy()
        codes = np.clip(codes, 0, int(info["cardinality"]) - 1)
        encoded.append(codes)
    return np.vstack(encoded).T.astype(np.int64), cardinalities

train_df = exploration.loc[train_mask]
valid_df = exploration.loc[valid_mask]

x_cont_train, cont_mean, cont_std = encode_continuous(train_df, continuous_cols)
x_cont_valid, _, _ = encode_continuous(valid_df, continuous_cols, cont_mean, cont_std)
x_bin_train = encode_binary(train_df, binary_cols)
x_bin_valid = encode_binary(valid_df, binary_cols)
x_cat_train, categorical_cardinalities = encode_categorical(train_df, categorical_cols)
x_cat_valid, _ = encode_categorical(valid_df, categorical_cols)
y_train = train_df[label_cols].to_numpy(dtype=np.float32)
y_valid = valid_df[label_cols].to_numpy(dtype=np.float32)

print("x_cont_train:", x_cont_train.shape)
print("x_bin_train:", x_bin_train.shape)
print("x_cat_train:", x_cat_train.shape)
print("target train means:", dict(zip(label_cols, y_train.mean(axis=0).round(4))))

x_cont_train: (2843200, 22)
x_bin_train: (2843200, 6)
x_cat_train: (2843200, 35)
target train means: {'y_complete': np.float32(0.3378), 'y_long': np.float32(0.2119), 'y_rewatch': np.float32(0.1015), 'y_neg': np.float32(0.1629)}


## 4. Behavior-Head MLP

The model predicts four simulated behavior outcomes for each user-video edge:

$$
\widehat y_{ij} =
(\widehat p_{complete,ij},\widehat p_{long,ij},\widehat p_{rewatch,ij},\widehat p_{neg,ij}).
$$

The network combines standardized continuous features, binary indicators, and categorical embeddings, then trains with binary cross-entropy over the four heads.

Early stopping monitors validation BCE loss. Training stops when validation loss does not improve by at least `MIN_VALID_LOSS_DELTA` for `EARLY_STOPPING_PATIENCE` epochs.


In [5]:
class EdgeFeatureEncoder(nn.Module):
    def __init__(self, n_cont, n_bin, categorical_cardinalities, emb_dim_cap=32):
        super().__init__()
        self.n_cont = int(n_cont)
        self.n_bin = int(n_bin)
        self.embeddings = nn.ModuleList()
        self.embedding_dims = []
        for cardinality in categorical_cardinalities:
            dim = min(emb_dim_cap, max(4, int(round(math.sqrt(cardinality)))))
            self.embeddings.append(nn.Embedding(int(cardinality), dim))
            self.embedding_dims.append(dim)
        self.output_dim = self.n_cont + self.n_bin + sum(self.embedding_dims)

    def forward(self, x_cont_batch, x_bin_batch, x_cat_batch):
        parts = []
        if self.n_cont:
            parts.append(x_cont_batch.float())
        if self.n_bin:
            parts.append(x_bin_batch.float())
        for j, emb in enumerate(self.embeddings):
            codes = x_cat_batch[:, j].long().clamp(min=0, max=emb.num_embeddings - 1)
            parts.append(emb(codes))
        return torch.cat(parts, dim=1)

class BaselineBehaviorMLP(nn.Module):
    def __init__(self, n_cont, n_bin, categorical_cardinalities, hidden_sizes, dropout=0.1, out_dim=4):
        super().__init__()
        self.encoder = EdgeFeatureEncoder(n_cont, n_bin, categorical_cardinalities)
        layers = []
        dim = self.encoder.output_dim
        for hidden in hidden_sizes:
            layers.append(nn.Linear(dim, hidden))
            layers.append(nn.SiLU())
            layers.append(nn.Dropout(dropout))
            dim = hidden
        layers.append(nn.Linear(dim, out_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_cont_batch, x_bin_batch, x_cat_batch):
        return self.mlp(self.encoder(x_cont_batch, x_bin_batch, x_cat_batch))

def make_tensor_dataset(x_cont, x_bin, x_cat, y=None):
    tensors = [
        torch.from_numpy(x_cont).float(),
        torch.from_numpy(x_bin).float(),
        torch.from_numpy(x_cat).long(),
    ]
    if y is not None:
        tensors.append(torch.from_numpy(y).float())
    return TensorDataset(*tensors)

train_ds = make_tensor_dataset(x_cont_train, x_bin_train, x_cat_train, y_train)
valid_ds = make_tensor_dataset(x_cont_valid, x_bin_valid, x_cat_valid, y_valid)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)

model = BaselineBehaviorMLP(
    n_cont=len(continuous_cols),
    n_bin=len(binary_cols),
    categorical_cardinalities=categorical_cardinalities,
    hidden_sizes=HIDDEN_SIZES,
    dropout=DROPOUT,
    out_dim=len(label_cols),
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()

print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

BaselineBehaviorMLP(
  (encoder): EdgeFeatureEncoder(
    (embeddings): ModuleList(
      (0): Embedding(4, 4)
      (1): Embedding(241, 16)
      (2): Embedding(5, 4)
      (3): Embedding(9, 4)
      (4-6): 3 x Embedding(8, 4)
      (7): Embedding(3, 4)
      (8): Embedding(9, 4)
      (9): Embedding(31, 6)
      (10): Embedding(1077, 32)
      (11): Embedding(14, 4)
      (12): Embedding(11, 4)
      (13): Embedding(4, 4)
      (14): Embedding(48, 7)
      (15): Embedding(341, 18)
      (16): Embedding(8, 4)
      (17): Embedding(6, 4)
      (18-24): 7 x Embedding(4, 4)
      (25): Embedding(7408, 32)
      (26): Embedding(3, 4)
      (27): Embedding(19, 4)
      (28): Embedding(3, 4)
      (29): Embedding(7664, 32)
      (30): Embedding(547, 23)
      (31): Embedding(511, 23)
      (32): Embedding(40, 6)
      (33): Embedding(138, 12)
      (34): Embedding(215, 15)
    )
  )
  (mlp): Sequential(
    (0): Linear(in_features=342, out_features=256, bias=True)
    (1): SiLU()
    (2): D

In [6]:
@torch.no_grad()
def evaluate_model(loader):
    model.eval()
    total_loss = 0.0
    total_n = 0
    logits_all = []
    y_all = []
    for xb_cont, xb_bin, xb_cat, yb in loader:
        xb_cont = xb_cont.to(DEVICE)
        xb_bin = xb_bin.to(DEVICE)
        xb_cat = xb_cat.to(DEVICE)
        yb = yb.to(DEVICE)
        logits = model(xb_cont, xb_bin, xb_cat)
        loss = criterion(logits, yb)
        total_loss += float(loss.detach().cpu()) * len(yb)
        total_n += len(yb)
        logits_all.append(logits.detach().cpu())
        y_all.append(yb.detach().cpu())
    logits_np = torch.cat(logits_all).numpy()
    y_np = torch.cat(y_all).numpy()
    prob_np = 1.0 / (1.0 + np.exp(-logits_np))
    auc_by_head = {}
    for j, name in enumerate(label_cols):
        if len(np.unique(y_np[:, j])) < 2:
            auc_by_head[name] = None
        else:
            auc_by_head[name] = float(roc_auc_score(y_np[:, j], prob_np[:, j]))
    valid_aucs = [v for v in auc_by_head.values() if v is not None]
    return {
        "loss": total_loss / max(total_n, 1),
        "auc_by_head": auc_by_head,
        "mean_auc": float(np.mean(valid_aucs)) if valid_aucs else None,
    }

training_history = []
best_state = None
best_valid_loss = np.inf
best_epoch = 0
epochs_without_improvement = 0
early_stopped = False

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_n = 0
    for xb_cont, xb_bin, xb_cat, yb in train_loader:
        xb_cont = xb_cont.to(DEVICE)
        xb_bin = xb_bin.to(DEVICE)
        xb_cat = xb_cat.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb_cont, xb_bin, xb_cat)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += float(loss.detach().cpu()) * len(yb)
        running_n += len(yb)

    valid_metrics = evaluate_model(valid_loader)
    improved = valid_metrics["loss"] < (best_valid_loss - MIN_VALID_LOSS_DELTA)
    if improved:
        best_valid_loss = valid_metrics["loss"]
        best_epoch = epoch
        epochs_without_improvement = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_without_improvement += 1

    record = {
        "epoch": epoch,
        "train_loss": running_loss / running_n,
        "valid_loss": valid_metrics["loss"],
        "valid_mean_auc": valid_metrics["mean_auc"],
        "valid_auc_by_head": valid_metrics["auc_by_head"],
        "best_valid_loss": best_valid_loss,
        "best_epoch": best_epoch,
        "improved": bool(improved),
        "epochs_without_improvement": epochs_without_improvement,
    }
    training_history.append(record)
    print(record)

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        early_stopped = True
        print(
            f"Early stopping at epoch {epoch}: validation BCE loss did not improve "
            f"for {EARLY_STOPPING_PATIENCE} epochs. Best epoch = {best_epoch}, "
            f"best validation BCE = {best_valid_loss:.6f}."
        )
        break

if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
final_valid_metrics = evaluate_model(valid_loader)
print("best epoch:", best_epoch)
print("best validation BCE loss:", best_valid_loss)
print("final valid metrics after restoring best state:", final_valid_metrics)


{'epoch': 1, 'train_loss': 0.4449590800166734, 'valid_loss': 0.41114823071421297, 'valid_mean_auc': 0.7447814506021759, 'valid_auc_by_head': {'y_complete': 0.8165509691095194, 'y_long': 0.7164588391868151, 'y_rewatch': 0.7767011323987276, 'y_neg': 0.6694148617136414}}


{'epoch': 2, 'train_loss': 0.4108730397408238, 'valid_loss': 0.4060335000479027, 'valid_mean_auc': 0.7535963098761826, 'valid_auc_by_head': {'y_complete': 0.8217126708641782, 'y_long': 0.7249186668813572, 'y_rewatch': 0.7848289061855744, 'y_neg': 0.6829249955736207}}


{'epoch': 3, 'train_loss': 0.4063251519109268, 'valid_loss': 0.4036560963906796, 'valid_mean_auc': 0.7582773148223244, 'valid_auc_by_head': {'y_complete': 0.8237847923782967, 'y_long': 0.7291262294944977, 'y_rewatch': 0.7884346964506417, 'y_neg': 0.6917635409658619}}


{'epoch': 4, 'train_loss': 0.40405939733371615, 'valid_loss': 0.40235533801101875, 'valid_mean_auc': 0.7606214297416515, 'valid_auc_by_head': {'y_complete': 0.8244542883938863, 'y_long': 0.7304158540905573, 'y_rewatch': 0.7905941556533654, 'y_neg': 0.6970214208287969}}


{'epoch': 5, 'train_loss': 0.40252346090205143, 'valid_loss': 0.4016515649726838, 'valid_mean_auc': 0.7622800349371589, 'valid_auc_by_head': {'y_complete': 0.8249406762181781, 'y_long': 0.7313077099871101, 'y_rewatch': 0.7915988335749182, 'y_neg': 0.7012729199684293}}


final valid metrics: {'loss': 0.4016515649726838, 'auc_by_head': {'y_complete': 0.8249406762181781, 'y_long': 0.7313077099871101, 'y_rewatch': 0.7915988335749182, 'y_neg': 0.7012729199684293}, 'mean_auc': 0.7622800349371589}


## 5. Holdout Scoring

After training, the model predicts behavior probabilities for every holdout candidate. These probabilities are converted into a single vanilla recommender score using the calibrated head weights:

$$
S^{vanilla}_{ij}
= w_c\widehat p_{complete,ij}
+ w_l\widehat p_{long,ij}
+ w_r\widehat p_{rewatch,ij}
+ w_n\widehat p_{neg,ij}.
$$

The score is used only for ranking. True utility `u_star` is kept for evaluation after the ranking is produced.


In [7]:
with open(HEAD_WEIGHT_PATH) as f:
    head_weight_payload = json.load(f)
head_weights = head_weight_payload.get("weights", {})
w_complete = float(head_weights.get("w_complete", 1.0))
w_long = float(head_weights.get("w_long", 0.0))
w_rewatch = float(head_weights.get("w_rewatch", 0.0))
w_neg = float(head_weights.get("w_neg", -1.0))
print("head weights:", {"w_complete": w_complete, "w_long": w_long, "w_rewatch": w_rewatch, "w_neg": w_neg})

x_cont_holdout, _, _ = encode_continuous(holdout, continuous_cols, cont_mean, cont_std)
x_bin_holdout = encode_binary(holdout, binary_cols)
x_cat_holdout, _ = encode_categorical(holdout, categorical_cols)
holdout_ds = make_tensor_dataset(x_cont_holdout, x_bin_holdout, x_cat_holdout, y=None)
holdout_loader = DataLoader(holdout_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)

@torch.no_grad()
def predict_probs(loader):
    model.eval()
    chunks = []
    for xb_cont, xb_bin, xb_cat in loader:
        xb_cont = xb_cont.to(DEVICE)
        xb_bin = xb_bin.to(DEVICE)
        xb_cat = xb_cat.to(DEVICE)
        logits = model(xb_cont, xb_bin, xb_cat)
        probs = torch.sigmoid(logits).detach().cpu().numpy().astype(np.float32)
        chunks.append(probs)
    return np.vstack(chunks)

holdout_probs = predict_probs(holdout_loader)
score_vanilla = (
    w_complete * holdout_probs[:, 0]
    + w_long * holdout_probs[:, 1]
    + w_rewatch * holdout_probs[:, 2]
    + w_neg * holdout_probs[:, 3]
).astype(np.float32)

holdout_scores = holdout[["user_id", "video_id", "u_star"]].copy()
holdout_scores["p_complete_hat"] = holdout_probs[:, 0]
holdout_scores["p_long_hat"] = holdout_probs[:, 1]
holdout_scores["p_rewatch_hat"] = holdout_probs[:, 2]
holdout_scores["p_neg_hat"] = holdout_probs[:, 3]
holdout_scores["score_vanilla"] = score_vanilla
holdout_scores.to_parquet(HOLDOUT_SCORE_OUTPUT_PATH, index=False)
print("saved holdout scores:", HOLDOUT_SCORE_OUTPUT_PATH)
display(holdout_scores.head())

head weights: {'w_complete': 0.05088338408773398, 'w_long': 0.014799267862244149, 'w_rewatch': -0.05711003386925411, 'w_neg': -0.3119390730200324}


saved holdout scores: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation/validation_baseline_holdout_scores.parquet


,user_id,video_id,u_star,p_complete_hat,p_long_hat,p_rewatch_hat,p_neg_hat,score_vanilla
0,3,3272,3.137872,0.205800,0.100048,0.016492,0.036457,-0.000362
1,3,6247,1.543000,0.007695,0.157184,0.002050,0.079830,-0.022301
2,3,195,2.392009,0.711614,0.516063,0.321250,0.174965,-0.029078
3,3,8057,3.419580,0.708209,0.536039,0.353782,0.184813,-0.033886
4,3,8885,3.135083,0.523458,0.133433,0.029311,0.042108,0.013801


## 6. Top-100 Recommendations

For each user, the notebook sorts the 1,000 holdout candidates by `score_vanilla` and stores the top 100 videos. It then reports the realized true utility of those recommendations:

$$
\mathrm{Utility@100}
= \frac{1}{|I|}\sum_i \frac{1}{100}\sum_{j\in R_i(100)} u^*_{ij}.
$$

This is a validation metric, not a training objective. It tells us how much true simulated utility the behavior-based baseline delivers.


In [8]:
def topk_by_score(frame, score_col, k=TOP_K):
    return (
        frame.sort_values(["user_id", score_col], ascending=[True, False], kind="mergesort")
        .groupby("user_id", group_keys=False, sort=False)
        .head(k)
        .copy()
    )

def utility_by_user(frame, method, k=TOP_K):
    out = frame.groupby("user_id", observed=True)["u_star"].mean().reset_index(name="utility_at_100")
    out["method"] = method
    return out

top100 = topk_by_score(holdout_scores, "score_vanilla", TOP_K)
top100["rank"] = top100.groupby("user_id")["score_vanilla"].rank(method="first", ascending=False).astype("int32")
top100 = top100[["user_id", "video_id", "rank", "score_vanilla", "u_star"]].sort_values(["user_id", "rank"]).reset_index(drop=True)
top100.to_parquet(TOP100_OUTPUT_PATH, index=False)

oracle_top100 = topk_by_score(holdout_scores, "u_star", TOP_K)

rng = np.random.default_rng(RANDOM_SEED + 100)
random_top100 = (
    holdout_scores.assign(_rand=rng.random(len(holdout_scores)))
    .sort_values(["user_id", "_rand"], ascending=[True, True], kind="mergesort")
    .groupby("user_id", group_keys=False, sort=False)
    .head(TOP_K)
    .drop(columns=["_rand"])
)

per_user = pd.concat([
    utility_by_user(random_top100, "random"),
    utility_by_user(top100, "vanilla_mlp"),
    utility_by_user(oracle_top100, "oracle"),
], ignore_index=True)

oracle_by_user = per_user.loc[per_user["method"].eq("oracle"), ["user_id", "utility_at_100"]].rename(columns={"utility_at_100": "oracle_utility_at_100"})
per_user = per_user.merge(oracle_by_user, on="user_id", how="left", validate="many_to_one")
per_user["regret_at_100"] = per_user["oracle_utility_at_100"] - per_user["utility_at_100"]

summary_rows = []
for method, group in per_user.groupby("method", sort=False):
    summary_rows.append({
        "method": method,
        "utility_at_100": float(group["utility_at_100"].mean()),
        "regret_at_100": float(group["regret_at_100"].mean()),
        "se": float(group["utility_at_100"].std(ddof=1) / np.sqrt(group["user_id"].nunique())),
        "users": int(group["user_id"].nunique()),
    })
summary = pd.DataFrame(summary_rows)
method_order = {"random": 0, "vanilla_mlp": 1, "oracle": 2}
summary = summary.sort_values("method", key=lambda s: s.map(method_order)).reset_index(drop=True)

print("saved top100:", TOP100_OUTPUT_PATH)
display(summary)

saved top100: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation/validation_baseline_top100.parquet


,method,utility_at_100,regret_at_100,se,users
0,random,1.951982,1.964269,0.036329,1777
1,vanilla_mlp,2.249415,1.666837,0.040961,1777
2,oracle,3.916251,0.000000,0.038757,1777


## 7. Save Baseline Artifacts

This section saves all outputs needed for later comparison:

| Output | Contents |
|---|---|
| `validation_baseline_mlp.pt` | trained MLP weights and preprocessing metadata |
| `validation_baseline_holdout_scores.parquet` | one row per holdout candidate with predicted behavior probabilities and `score_vanilla` |
| `validation_baseline_top100.parquet` | top-100 recommendation list for each user |
| `validation_baseline_metrics.json` | training history, validation AUC/BCE, score weights, and top-100 utility diagnostics |


In [9]:
metrics = {
    "random_seed": RANDOM_SEED,
    "device": DEVICE,
    "num_epochs": NUM_EPOCHS,
    "num_epochs_run": len(training_history),
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "min_valid_loss_delta": MIN_VALID_LOSS_DELTA,
    "early_stopped": bool(early_stopped),
    "best_epoch": int(best_epoch),
    "best_validation_bce_loss": float(best_valid_loss),
    "batch_size": BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "hidden_sizes": HIDDEN_SIZES,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "train_rows": int(train_mask.sum()),
    "valid_rows": int(valid_mask.sum()),
    "holdout_rows": int(len(holdout_scores)),
    "num_users": int(holdout_scores["user_id"].nunique()),
    "holdout_candidates_per_user": int(holdout_scores.groupby("user_id").size().median()),
    "top_k": TOP_K,
    "validation_bce_loss": float(final_valid_metrics["loss"]),
    "validation_auc_by_head": final_valid_metrics["auc_by_head"],
    "validation_mean_auc": final_valid_metrics["mean_auc"],
    "training_history": training_history,
    "head_weights_path": str(HEAD_WEIGHT_PATH),
    "head_weights": {"w_complete": w_complete, "w_long": w_long, "w_rewatch": w_rewatch, "w_neg": w_neg},
    "summary_table": summary.to_dict(orient="records"),
    "feature_columns_used": feature_cols,
    "continuous_cols": continuous_cols,
    "binary_cols": binary_cols,
    "categorical_cols": categorical_cols,
    "dropped_features": sorted(DROP_FEATURES),
    "forbidden_features": sorted(FORBIDDEN_FEATURES),
    "outputs": {
        "holdout_scores": str(HOLDOUT_SCORE_OUTPUT_PATH),
        "top100": str(TOP100_OUTPUT_PATH),
        "metrics": str(METRICS_OUTPUT_PATH),
        "model": str(MODEL_OUTPUT_PATH),
    },
}
METRICS_OUTPUT_PATH.write_text(json.dumps(metrics, indent=2))

torch.save({
    "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "config": {
        "continuous_cols": continuous_cols,
        "binary_cols": binary_cols,
        "categorical_cols": categorical_cols,
        "categorical_cardinalities": categorical_cardinalities,
        "hidden_sizes": HIDDEN_SIZES,
        "dropout": DROPOUT,
        "head_weights": metrics["head_weights"],
        "max_epochs": NUM_EPOCHS,
        "num_epochs_run": len(training_history),
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "min_valid_loss_delta": MIN_VALID_LOSS_DELTA,
        "early_stopped": bool(early_stopped),
        "best_epoch": int(best_epoch),
        "best_validation_bce_loss": float(best_valid_loss),
    },
    "preprocessing": {
        "cont_mean": cont_mean,
        "cont_std": cont_std,
    },
}, MODEL_OUTPUT_PATH)

print("saved metrics:", METRICS_OUTPUT_PATH)
print("saved model:", MODEL_OUTPUT_PATH)
print(json.dumps({
    "validation_bce_loss": metrics["validation_bce_loss"],
    "validation_mean_auc": metrics["validation_mean_auc"],
    "num_epochs_run": metrics["num_epochs_run"],
    "early_stopped": metrics["early_stopped"],
    "best_epoch": metrics["best_epoch"],
    "best_validation_bce_loss": metrics["best_validation_bce_loss"],
    "summary_table": metrics["summary_table"],
}, indent=2))

saved metrics: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation/validation_baseline_metrics.json
saved model: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation/validation_baseline_mlp.pt
{
  "validation_bce_loss": 0.4016515649726838,
  "validation_mean_auc": 0.7622800349371589,
  "summary_table": [
    {
      "method": "random",
      "utility_at_100": 1.951982388513334,
      "regret_at_100": 1.9642690847753062,
      "se": 0.03632921721552027,
      "users": 1777
    },
    {
      "method": "vanilla_mlp",
      "utility_at_100": 2.2494149390239984,
      "regret_at_100": 1.6668365342646418,
      "se": 0.04096090592953062,
      "users": 1777
    },
    {
      "method": "oracle",
      "utility_at_100": 3.9162514732886406,
      "regret_at_100": 0.0,
      "se": 0.038756555085069755,
      "users": 1777
    }
  ]
}
